# ElasticSearchへの登録、ベクトル検索のシンプルなサンプル

ElasticSearchクライアントを使わず、requestsを使い、REST APIへリクエストすることでElasticSearchを扱う

※ElasticSearchのバージョンアップ後、ElasticSearchクライアントの対応まで時間がかかることがあり、REST APIでの操作を推奨する旨、Elastic社のサポートチームやコミュニティでも議論されているため。

## 必要パッケージのインポート

In [ ]:
import requests
# from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import json
import time

## 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test01'
HEADERS = {
    'Accept': 'application/vnd.elasticsearch+json; compatible-with=8',
    'Content-Type': 'application/vnd.elasticsearch+json; compatible-with=8',
}

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## インデックスの有無チェック、あれば削除

In [ ]:
res = requests.head(f"{ES_URL}/{INDEX_NAME}")
exists_index = res.status_code == 200
exists_index

In [ ]:
if exists_index:
    res = requests.delete(f"{ES_URL}/{INDEX_NAME}/", headers=HEADERS,)
    print(f'index {INDEX_NAME} is deleted.')
else:
    print(f'index {INDEX_NAME} does not exist.')

## インデックス作成

In [ ]:
mapping = {
    'mappings': {
        'properties': {
            'text': {'type': 'text'},
            'vector': {
                'type': 'dense_vector',
                'dims': MODEL_DIM,
                'index': True,
                'similarity': 'cosine'
            }
        }
    }
}
res = requests.put(f"{ES_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping))
res.raise_for_status()
print(f"Index '{INDEX_NAME}' created.")

## ドキュメントをインデックス

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

for text in texts:
    vector = model.encode(text)
    doc = {
        "text": text,
        "vector": vector.tolist()
    }
    res = requests.post(f"{ES_URL}/{INDEX_NAME}/_doc", headers=HEADERS, data=json.dumps(doc))
    res.raise_for_status()
    print(f"Indexed: {text}")

## ベクトル検索

In [ ]:
QUERY_TEXT = '日本の都市'
TOP_K = 3

In [ ]:
# インデックス登録後、検索できるようになるまでにタイムラグが生じることがあるため、3秒ほど待つ
time.sleep(3)

In [ ]:
query_vector = model.encode(QUERY_TEXT)
query = {
    'knn': {
        'field': 'vector',
        'query_vector': query_vector.tolist(),
        'k': TOP_K,
        'num_candidates': 100
    }
}
body = {
    'size': TOP_K,
    'query': query
}
res = requests.post(f"{ES_URL}/{INDEX_NAME}/_search", headers=HEADERS, data=json.dumps(body))
res.raise_for_status()

hits = res.json()['hits']['hits']
print("--- 検索結果 ---")
for hit in hits:
    print(f"スコア: {hit['_score']:.4f}, テキスト: {hit['_source']['text']}")